# 语言模型零基础 02：平滑、回退、OOV 与困惑度

第一课回答“Bigram 怎么数、句子概率怎么算”。这一课回答工程上更重要的问题：**遇到训练集没见过的搭配或词时怎么办，以及怎样比较两个语言模型。**

完成本课后，你应该能够：

1. 区分平滑、回退和插值；
2. 解释为什么 Add-k 适合教学但通常不是最佳工程方案；
3. 正确处理 OOV 与 `<unk>`；
4. 计算并解释 perplexity（困惑度）；
5. 看懂 ARPA 文件中概率和 backoff weight 的基本角色。


## 学习规则

每个实验先预测结果，再运行。只修改一个参数，并用一句话记录：**我原来预测什么、实际发生什么、为什么。**

> 本课自己实现的是教学模型。真实工具中的 Modified Kneser–Ney、ARPA backoff 和 OpenFst 编码会更严格，后续课程会逐层替换教学简化。


In [ ]:
from collections import Counter
from math import exp, isclose, log, sqrt

START = "<s>"
END = "</s>"
UNK = "<unk>"

train_corpus = [
    "我 爱 自然 语言",
    "我 爱 语音 识别",
    "你 爱 自然 语言",
    "我 学习 语音 识别",
    "你 学习 语言 模型",
    "我 学习 语言 模型",
    "他 研究 声学 模型",
]

def tokenize(sentence):
    return sentence.strip().split()

print("训练句数：", len(train_corpus))
for sentence in train_corpus:
    print("  ", tokenize(sentence))


## 1. 同时训练 Unigram、Bigram 和 Trigram

- Unigram：`P(词)`，不看历史；
- Bigram：`P(词 | 前 1 个词)`；
- Trigram：`P(词 | 前 2 个词)`。

阶数越高，上下文越具体，但数据也越稀疏。Trigram 没见过，不代表句子不合理。


In [ ]:
vocabulary = sorted({word for sentence in train_corpus for word in tokenize(sentence)} | {END})

def count_ngrams(sentences, order):
    context_counts = Counter()
    ngram_counts = Counter()
    for sentence in sentences:
        seq = [START] * (order - 1) + tokenize(sentence) + [END]
        for i in range(order - 1, len(seq)):
            history = tuple(seq[i - order + 1:i]) if order > 1 else ()
            word = seq[i]
            context_counts[history] += 1
            ngram_counts[history, word] += 1
    return context_counts, ngram_counts

models = {order: count_ngrams(train_corpus, order) for order in (1, 2, 3)}

def ngram_probability(word, history, order, k=0.0):
    context_counts, ngram_counts = models[order]
    history = tuple(history[-(order - 1):]) if order > 1 else ()
    denominator = context_counts[history] + k * len(vocabulary)
    if denominator == 0:
        return 0.0
    return (ngram_counts[history, word] + k) / denominator

history = ["我", "爱"]
for word in ["自然", "语音", "语言"]:
    print(f"P({word} | 我 爱) = {ngram_probability(word, history, 3):.3f}")


## 2. 三种容易混淆的处理

### 平滑（smoothing）
重新分配概率质量，让未见事件不再必然为 0。Add-k 是最直观的例子。

### 回退（backoff）
如果高阶 N-gram 没有可靠统计，就退到低一阶。例如 Trigram 不可用时查 Bigram，再不行查 Unigram。严格 ARPA 回退还需要 backoff weight，保证概率分布正确归一化。

### 插值（interpolation）
不管高阶是否出现，都同时混合多个阶：

$$P(w|h)=\lambda_3P_3(w|h)+\lambda_2P_2(w|h)+\lambda_1P_1(w)$$

其中 $\lambda_1+\lambda_2+\lambda_3=1$。


In [ ]:
def teaching_backoff_probability(word, history):
    """只展示“逐阶退回”的直觉；不是归一化的生产 ARPA backoff。"""
    for order in (3, 2):
        p = ngram_probability(word, history, order, k=0.0)
        if p > 0:
            return p, order
    return ngram_probability(word, history, 1, k=0.1), 1

for word in ["自然", "语言", "模型"]:
    probability, used_order = teaching_backoff_probability(word, ["我", "爱"])
    print(f"预测 {word:2s}: 使用 {used_order}-gram，概率={probability:.4f}")


In [ ]:
def interpolated_probability(word, history, lambdas=(0.2, 0.3, 0.5)):
    lambda1, lambda2, lambda3 = lambdas
    if not isclose(sum(lambdas), 1.0, abs_tol=1e-9):
        raise ValueError("三个 lambda 必须相加等于 1")
    p1 = ngram_probability(word, history, 1, k=0.01)
    p2 = ngram_probability(word, history, 2, k=0.0)
    p3 = ngram_probability(word, history, 3, k=0.0)
    return lambda1 * p1 + lambda2 * p2 + lambda3 * p3

for word in ["自然", "语言", "模型"]:
    print(f"P_interp({word} | 我 爱) = {interpolated_probability(word, ['我', '爱']):.4f}")


### 插值滑块实验

拖动 $\lambda_3$ 和 $\lambda_2$；$\lambda_1$ 自动使用剩余权重。观察高阶权重增大时，见过和未见过的 Trigram 如何变化。


In [ ]:
import ipywidgets as widgets

def interpolation_demo(lambda3=0.5, lambda2=0.3):
    lambda1 = 1.0 - lambda2 - lambda3
    if lambda1 < 0:
        print("lambda2 + lambda3 不能超过 1")
        return
    lambdas = (lambda1, lambda2, lambda3)
    print(f"lambda1={lambda1:.1f}, lambda2={lambda2:.1f}, lambda3={lambda3:.1f}")
    for word in ["自然", "语言", "模型"]:
        p = interpolated_probability(word, ["我", "爱"], lambdas)
        print(f"P({word} | 我 爱) = {p:.4f}")

widgets.interact(
    interpolation_demo,
    lambda3=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.1),
    lambda2=widgets.FloatSlider(value=0.3, min=0, max=1, step=0.1),
);


## 3. 困惑度 Perplexity

对含 $N$ 个预测目标的测试序列：

$$PPL=\exp\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(w_i|h_i)\right)$$

直觉：模型平均每一步像是在多少个候选中犹豫。**在相同测试集、词表和 tokenization 下，PPL 越低通常越好。**不同设置的 PPL 不应直接横比。


In [ ]:
def sentence_log_probability(sentence, lambdas=(0.2, 0.3, 0.5)):
    seq = [START, START, *tokenize(sentence), END]
    total = 0.0
    predicted_tokens = 0
    for i in range(2, len(seq)):
        word = seq[i]
        if word not in vocabulary:
            return float("-inf"), 0
        p = interpolated_probability(word, seq[:i], lambdas)
        if p <= 0:
            return float("-inf"), 0
        total += log(p)
        predicted_tokens += 1
    return total, predicted_tokens

def corpus_perplexity(sentences, lambdas=(0.2, 0.3, 0.5)):
    total_logp = 0.0
    total_tokens = 0
    for sentence in sentences:
        logp, count = sentence_log_probability(sentence, lambdas)
        if count == 0:
            return float("inf")
        total_logp += logp
        total_tokens += count
    return exp(-total_logp / total_tokens)

validation = ["我 爱 语言 模型", "你 爱 语音 识别"]
for lambdas in [(0.6, 0.3, 0.1), (0.2, 0.3, 0.5), (0.1, 0.2, 0.7)]:
    print(lambdas, f"PPL={corpus_perplexity(validation, lambdas):.3f}")


### 用验证集选择插值权重

权重不能看测试集答案来选，否则会产生数据泄漏。下面用小型 validation set 做网格搜索。真实项目会使用更细的搜索或专门训练方法。


In [ ]:
candidates = []
for i in range(11):
    lambda1 = i / 10
    for j in range(11 - i):
        lambda2 = j / 10
        lambda3 = 1.0 - lambda1 - lambda2
        lambdas = (lambda1, lambda2, lambda3)
        candidates.append((corpus_perplexity(validation, lambdas), lambdas))

for ppl, lambdas in sorted(candidates)[:5]:
    print(f"PPL={ppl:.3f}  lambdas(1,2,3)={tuple(round(x, 1) for x in lambdas)}")


## 4. OOV 与 `<unk>`

OOV（out-of-vocabulary）是模型词表外的词。平滑只能帮助“词表内但搭配没见过”的情况；一个根本不在词表里的新词，需要单独策略。

常见做法：

1. 训练时把低频词替换成 `<unk>`，让模型学到 `<unk>` 的概率；
2. 测试时把词表外词映射为 `<unk>`；
3. 或使用字、subword/token 单元，降低整词 OOV。


In [ ]:
raw_word_counts = Counter(word for sentence in train_corpus for word in tokenize(sentence))
rare_words = {word for word, count in raw_word_counts.items() if count <= 1}

def replace_rare_for_training(sentence):
    return " ".join(UNK if word in rare_words else word for word in tokenize(sentence))

known_after_replacement = (set(raw_word_counts) - rare_words) | {UNK}

def map_oov_at_test_time(sentence):
    return " ".join(word if word in known_after_replacement else UNK for word in tokenize(sentence))

print("低频词：", sorted(rare_words))
print("训练替换：", replace_rare_for_training("他 研究 声学 模型"))
print("测试映射：", map_oov_at_test_time("大家 研究 OpenFst 模型"))


## 5. ARPA 文件预览

传统 N-gram 工具常输出 ARPA 文本，大致结构如下：

```text
\data\
ngram 1=...
ngram 2=...

\1-grams:
log10概率    词       backoff权重
-0.6021      我       -0.1761

\2-grams:
log10概率    前词 后词  backoff权重
-0.3010      我 爱     -0.1249
```

ARPA 通常保存以 10 为底的 log probability，而 OpenFst 常用自然对数域中的代价。转换工具负责正确处理符号、回退弧和权重，不要仅靠字符串替换。


## 6. 自动判题

请先填写答案，再运行。不要查看已运行版来猜。


In [ ]:
# 请修改四个答案
answer_1 = None  # lambda3=0.5、lambda2=0.3 时，lambda1 是多少？
answer_2 = None  # 两步概率分别为 0.5 和 0.25 时，PPL 是多少？
answer_3 = ""    # 相同评测设置下，PPL 越 lower 还是 higher 越好？填英文
answer_4 = ""    # 常用于承接 OOV 概率的特殊 token 是什么？

checks = [
    answer_1 is not None and isclose(float(answer_1), 0.2, abs_tol=1e-9),
    answer_2 is not None and isclose(float(answer_2), sqrt(8), rel_tol=1e-6),
    str(answer_3).strip().lower() == "lower",
    str(answer_4).strip().lower() == "<unk>",
]
for i, ok in enumerate(checks, 1):
    print(("✅" if ok else "❌"), f"第 {i} 题")
print(f"得分：{sum(checks)}/4")
if all(checks):
    print("通过：下一课可以把语言模型变成状态和弧。")
else:
    print("未通过也没关系：回到对应实验，先解释现象再重算。")


<details><summary>完成后展开参考答案</summary>

1. `0.2`；三个权重之和必须为 1。  
2. $\exp[-(\log 0.5+\log 0.25)/2]=\sqrt{8}\approx2.828$。  
3. `lower`。  
4. `<unk>`。

</details>

## 离场票

不看上文，口头解释“未见 N-gram”和“OOV”为什么不是同一问题，并说明平滑、回退、插值各自做什么。

下一课：打开 `语言模型零基础_03_FSA_FST与第一张OpenFst图.ipynb`，把 Bigram 语言模型编译为真实 OpenFst 图并执行最短路径。
